In [ ]:
# ===========================================================================
# Cell 0: v18 setup. Run before anything else.
#
# v18 is not another frame inventory. v17's normalised MI between frame and sentence
# label is exactly 1.000 and a bag of words names the frame from masked text at 0.961
# over 19 classes, so the label is a deterministic function of something visible on the
# surface. That is the v13 composition shortcut with construction in place of pair count.
#
# The change is two axes. Assertions bind pairs and carry labels. Modifiers carry no
# label and are drawn independently. `regimen`, `title`, `appositive`, `coordinate` and
# `sequential` move to the modifier axis, on the evidence that appositive and coordinate
# are the main clause of zero corpus sentences out of 295 and regimen of 9 out of ~52
# appearances. Two variants make positive assertions impure: `unlisted` retargets the
# assertion at a referent that is not one of the listed entities, `list_target` expands
# the object into a list so a positive sentence carries several positive pairs.
#
# The check that matters here is label correctness, not F1. v18 deliberately writes
# sentences full of effect vocabulary that are gold NONE, which is exactly the shape of
# output that is worth nothing if the model does not honour it.
# ===========================================================================

import os
import json
import importlib
import re
import random
from collections import Counter, defaultdict

from openai import OpenAI

import ddi.prompt, ddi.prompt_v18, ddi.resolve, ddi.gates, ddi.synth, ddi.divergence
import ddi.frame_diag, ddi.pruning, ddi.verify_binary
for m in (ddi.synth, ddi.prompt, ddi.prompt_v18, ddi.resolve, ddi.gates,
          ddi.divergence, ddi.frame_diag, ddi.pruning, ddi.verify_binary):
    importlib.reload(m)

from ddi.data import build_human
from ddi.vocab import build_vocab
from ddi.prompt_v18 import (make_specs, render, make_sample_fn, fingerprint,
                            ASSERTIONS, MODIFIERS)
from ddi.resolve import v14_sample_to_instances, generation_records
from ddi.synth import generate_raw, build_dataset_from_raw, RAW
from ddi.manifest import load_dataset
from ddi import gates, divergence as dv, frame_diag

MODEL, API, EFFORT = "gpt-oss-120b", "responses", "low"
V14_ID = "20260807-123340-ff79db"
V15_ID = None          # fill in
V17_ID = "20260818-100023-0da98b"

client = OpenAI(base_url="http://api.llm.apps.os.dcs.gla.ac.uk/v1",
                api_key=os.environ["IDA_LLM_API_KEY"], max_retries=5, timeout=60.0)

vocab = build_vocab()
train, dev, val = build_human()
per_sent = Counter(r["sent_id"] for r in train)
train_f = [r for r in train if per_sent[r["sent_id"]] < 190]
v14, _ = load_dataset(V14_ID)
v17, _ = load_dataset(V17_ID)

print(f"prompt sha {fingerprint()}")
print(f"{len(ASSERTIONS)} assertions, weights sum "
      f"{sum(a['w'] for a in ASSERTIONS.values()):.3f}")
print(f"{len(MODIFIERS)} modifiers")
print(f"human filtered {len(train_f)}, v14 {len(v14)}, v17 {len(v17)}")

_s = make_specs(1, vocab=vocab, seed=0)[0]
assert "assertions" in _s and "modifiers" in _s, "make_specs is not v18"
assert "asserts" in _s, "resolve.py reads spec['asserts']; v18 must emit it"



In [ ]:
V15_ID = "20260816-005908-3d6539"

In [ ]:

# ===========================================================================
# Cell 1: spec statistics and render. Free, and it is where the tuning is decided.
#
# Three quantities cannot all be matched at once. Corpus mention counts, corpus modifier
# rates and the corpus pair positive rate of 0.162 are jointly inconsistent under
# one-positive-pair assertions. v17 hit 0.162 exactly by tuning n_positives_by_k while
# getting entity counts and construction rates wrong. P_LIST_TARGET is the honest lever
# because list objects are what produce multi-pair positives in the corpus, but it is set
# from judgement, not measurement.
#
# What to look for in the renders: the `unlisted` specs must not name a second listed
# drug anywhere in the says. If one does, that spec is a mislabelled positive waiting to
# happen and the substitution in _to_unlisted is not firing.
# ===========================================================================
specs = make_specs(3000, vocab=vocab, seed=0)

sent_pos = sum(1 for s in specs if s["positives"]) / len(specs)
n_pairs = sum(len(s["matrix"]) for s in specs)
pair_pos = sum(sum(1 for v in s["matrix"].values() if v != "NONE")
               for s in specs) / n_pairs
print(f"sentence positive rate {sent_pos:.3f}   (corpus sample ~0.40)")
print(f"pair positive rate     {pair_pos:.3f}   (corpus 0.162, v17 0.162)")
print(f"pairs per sentence     {n_pairs / len(specs):.2f}   (corpus 6.29, v17 4.28)")
print(f"assertions per sentence {Counter(len(s['assertions']) for s in specs)}")
print(f"variants {Counter(tuple(s['variants']) for s in specs).most_common(6)}")

lab = Counter()
for s in specs:
    for a in s["assertions"]:
        lab[(a, bool(s["positives"]))] += 1
print(f"\n{'assertion':<20} {'n':>5} {'sent_pos_rate':>14}")
pure = []
for a in sorted(ASSERTIONS):
    t = lab[(a, True)] + lab[(a, False)]
    if not t:
        continue
    r = lab[(a, True)] / t
    print(f"{a:<20} {t:>5} {r:>14.3f}")
    if r in (0.0, 1.0):
        pure.append(a)
print(f"\nlabel-pure assertions: {pure or 'none'}")
assert not pure, "an assertion is still label-pure; v18's whole point is that none are"

for want in [("unlisted",), ("list_target",), ()]:
    s = next((x for x in specs if tuple(x["variants"]) == want
              and len(x["assertions"]) == (2 if not want else 1)), None)
    if s is None:
        s = next(x for x in specs if tuple(x["variants"]) == want)
    print(f"\n{'-' * 66}\n{s['assertions']}  variants={s['variants']}  "
          f"modifiers={s['modifiers']}")
    print(render(s))
    print(f"  gold: {s['positives'] or 'all NONE'}")


In [ ]:
for s in specs[:200]:
    print(f"\n{'-' * 66}\n{s['assertions']}  variants={s['variants']}  "
          f"modifiers={s['modifiers']}")
    print(render(s))
    print(f"  gold: {s['positives'] or 'all NONE'}")

In [ ]:

# ===========================================================================
# Cell 2: v18 smoke, 300 specs. Read it before spending the full run.
#
# The adherence question is different from v17's. v17 asked whether the model could
# write each construction. v18 asks whether it honours the two variants, because both of
# them create sentences whose surface reads positive and whose gold is NONE:
#
#   unlisted   the assertion must target something not in the drug list. If the model
#              substitutes a listed drug the pair is really positive and gold says NONE,
#              which is a false negative and recall is already v17's weak axis at 0.403.
#   list_target every listed member must actually receive the relation. If the model
#              writes only the first, the rest are false positives.
#
# Both are measurable with the calibrated binary verifier rather than a regex. Its NONE
# recall is 0.920 and positive recall 0.948 on real corpus text, and synthetic sentences
# are simpler, so treat a disagreement rate near 0.08 as the floor.
# ===========================================================================
GEN = "v18-check-2"
specs = make_specs(300, vocab=vocab, seed=0)
print(f"prompt sha {fingerprint()}")

generate_raw(specs, make_sample_fn(client, model=MODEL, reasoning_effort=EFFORT,
                                   api=API),
             gen_id=GEN, max_workers=16)

did, stats = build_dataset_from_raw(
    GEN, resolver=v14_sample_to_instances, mode="markers",
    generator={"prompt_sha": fingerprint(), "model": MODEL, "version": "v18",
               "note": "assertions and modifiers on separate axes, unlisted and "
                       "list_target variants, role codes"},
    vocab_source=vocab.fingerprint(), seed=0)
print(stats["reject_reasons"])
inst, _ = load_dataset(did)


In [ ]:

# ---------------------------------------------------------------------------
# 2a. Verifier adherence, split by variant. This is the cell that decides whether v18
#     is safe to scale, and it costs 300 calls.
# ---------------------------------------------------------------------------
from ddi.verify_binary import build_batches, make_binary_verifier, load_verdicts

VGEN = f"{GEN}-verify"
generate_raw(build_batches(inst), make_binary_verifier(client, model=MODEL, reasoning_effort="low"),
             gen_id=VGEN, max_workers=16)
vdf = load_verdicts(VGEN)

variant_of = {f"synth:{GEN}:{s['spec_index']}": tuple(s["variants"]) or ("plain",)
              for s in specs}
assert_of = {f"synth:{GEN}:{s['spec_index']}": tuple(s["assertions"]) for s in specs}

rows = defaultdict(lambda: Counter())
for r in vdf.itertuples():
    v = variant_of.get(re.sub(r":s\d+$", "", r.sent_id))
    if v is None:
        continue
    rows[v[0]][("gold_pos" if r.gold_pos else "gold_none",
                "said_pos" if r.flagged else "said_none")] += 1

print(f"\n{'variant':<14} {'n_none':>7} {'flagged':>8} {'flag_rate':>10} "
      f"{'n_pos':>6} {'missed':>7}")
for v, c in sorted(rows.items()):
    n_none = c[("gold_none", "said_pos")] + c[("gold_none", "said_none")]
    n_pos = c[("gold_pos", "said_pos")] + c[("gold_pos", "said_none")]
    fr = c[("gold_none", "said_pos")] / max(n_none, 1)
    ms = c[("gold_pos", "said_none")] / max(n_pos, 1)
    print(f"{v:<14} {n_none:>7} {c[('gold_none', 'said_pos')]:>8} {fr:>10.3f} "
          f"{n_pos:>6} {ms:>7.3f}")
print("""
  unlisted flag rate near the 0.08 floor      -> the variant works, scale it
  unlisted flag rate above ~0.20              -> the model is naming a listed drug
     anyway. Drop P_UNLISTED_TARGET to 0.10 and rerun this cell before the full run;
     do not carry a false-negative source into a 6000-spec dataset.
  list_target missed rate above ~0.20         -> only the first member is getting the
     relation. Cap members at 1 in _to_list_target.
""")

# ---------------------------------------------------------------------------
# 2b. The structural gate. Free, and it is the number v18 exists to move.
# ---------------------------------------------------------------------------
inst_f, missing = frame_diag.attach_frames(inst, GEN)
print(f"{len(inst_f)} joined, {missing} unjoined")

pur = frame_diag.label_purity(inst_f)
rec = frame_diag.frame_recoverability(inst_f)
sc = frame_diag.shortcut_via_frame(inst_f)
lift, top = gates.shortcut_probe(inst_f)
print(f"nmi_sentence          {pur['nmi_sentence']:.3f}   (v17 1.000, target lower)")
print(f"nmi_pair              {pur['nmi_pair']:.3f}   (v17 0.240)")
print(f"frame recoverability  {rec['accuracy']:.3f}   (v17 0.961)")
print(f"lift from frame alone {sc['lift_predicted_frame']:.3f}   (v17 0.183)")
print(f"direct probe lift     {lift:.3f}   (v17 0.240, corpus 0.135)")
print("\ntop probe features")
for name, coef in top:
    print(f"  {coef:>6.2f}  {name}")
print("""
  probe lift near 0.135 and features that are relation language rather than construction
  names -> the fix landed. If the features are still frame names the assertion axis is
  leaking and the modifier split did not go far enough.
""")

gates.report(inst_f, records=generation_records(GEN), strict=False)
print(dv.compare(train_f, inst, match_size=True).to_string(index=False))


In [ ]:
# ---------------------------------------------------------------------------
# Which list_target failure is it? Free, on data already on disk.
#
# Two explanations for missed 0.395 against plain's 0.208, and they need
# different fixes:
#
#   wording   "the same holds for" is read as subject-copy. The member is written
#             as a second agent doing the same thing, not as a second object the
#             first drug acts on. Gold marks agent-member; the sentence supports
#             member-object. Missed rate would then be flat in the number of
#             members, because the first member is already wrong.
#
#   capacity  the model writes the first member and drops the rest. Missed rate
#             would then rise with the number of members.
#
# Flat -> reword. Rising -> cap members. Both -> do both.
# ---------------------------------------------------------------------------
import json
import re
from collections import Counter, defaultdict

from ddi.verify_binary import load_verdicts
from ddi.synth import RAW

GEN, VGEN = "v18-check", "v18-check-verify"

n_members, variant = {}, {}
for line in (RAW / f"{GEN}.jsonl").read_text().splitlines():
    if not line:
        continue
    r = json.loads(line)
    if r.get("error") or not r.get("sample"):
        continue
    spec = r["spec"]
    sid = f"synth:{GEN}:{spec['spec_index']}"
    variant[sid] = tuple(spec["variants"]) or ("plain",)
    row = next((v for k, v in
                [x for g in spec["groups"] for x in g[1]]
                if k == "the same holds for"), None)
    n_members[sid] = len(re.findall(r"\{(\w)\}", row)) if row else 0

vdf = load_verdicts(VGEN)
buckets = defaultdict(Counter)
for r in vdf.itertuples():
    sid = re.sub(r":s\d+$", "", r.sent_id)
    v = variant.get(sid)
    if v is None or "list_target" not in v:
        continue
    buckets[n_members[sid]][("pos" if r.gold_pos else "none",
                             "flag" if r.flagged else "noflag")] += 1

print(f"{'members':>8} {'n_pos':>6} {'missed':>8} {'n_none':>7} {'flagged':>8}")
for m in sorted(buckets):
    c = buckets[m]
    npz = c[("pos", "flag")] + c[("pos", "noflag")]
    nn = c[("none", "flag")] + c[("none", "noflag")]
    print(f"{m:>8} {npz:>6} {c[('pos', 'noflag')] / max(npz, 1):>8.3f} "
          f"{nn:>7} {c[('none', 'flag')] / max(nn, 1):>8.3f}")
print("\nplain baseline: missed 0.208, flagged 0.009")

# ---------------------------------------------------------------------------
# Then read ten of them. The verifier at low effort has a plain-spec missed rate
# of 0.208 against a calibrated positive recall of 0.948, so it is unreliable in
# absolute terms and the reading is what settles it.
# ---------------------------------------------------------------------------
missed = [r.sent_id for r in vdf.itertuples()
          if r.gold_pos and not r.flagged
          and "list_target" in (variant.get(re.sub(r":s\d+$", "", r.sent_id)) or ())]
text = {}
for line in (RAW / f"{GEN}.jsonl").read_text().splitlines():
    if not line:
        continue
    r = json.loads(line)
    if r.get("sample"):
        text[f"synth:{GEN}:{r['spec']['spec_index']}"] = r["sample"]["sentence"]
import random 
# sample random 10 of the missed sentences and print them for inspection, with their specs
for sid in random.sample(missed, min(10, len(missed))):
    spec = next((s for s in specs if f"synth:{GEN}:{s['spec_index']}" == sid), None)
    print(f"\nSpec: {spec}")
    print("\n" + text.get(re.sub(r":s\d+$", "", sid), "?"))

In [ ]:
import sys
import requests

def run_masked_cluster_diagnostic():
    print("=== CLUSTER ENVIRONMENT TEST (WITH HEADERS) ===")
    print(f"Python Version: {sys.version.split()[0]}")
    
    # Define a clean, non-generic User-Agent string
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) GlasgowMScProject/1.0 (mailto:your_email@student.gla.ac.uk)"
    }
    
    # 1. Test External Network Access
    print("\n[1/2] Testing external internet connectivity with custom headers...")
    test_url = "https://nih.gov"
    
    try:
        res = requests.get(test_url, headers=headers, timeout=5)
        if res.status_code == 200:
            print(" -> SUCCESS: External API requests are working perfectly.")
            data = res.json()
            props = data["PropertyTable"]["Properties"][0]
            print(f"\n[2/2] Pipeline Test Successful:")
            print(f"    - InChIKey: {props.get('InChIKey')}")
            print(f"    - SMILES:   {props.get('SMILES')}")
        elif res.status_code == 403:
            print(" -> ERROR 403: Still forbidden. The cluster IP range itself might be blacklisted by NCBI.")
        else:
            print(f" -> ERROR: Returned unexpected status code {res.status_code}.")
    except Exception as e:
        print(f" -> CRITICAL ERROR: Network request failed. Context: {e}")

if __name__ == "__main__":
    run_masked_cluster_diagnostic()


In [ ]:
import json

# Generate a small sample of specs and run through the model with reasoning

sample_specs = make_specs(5, vocab=vocab, seed=42)

for spec in sample_specs:
    print(f"\n{'-' * 70}")
    print(f"Spec {spec['spec_index']}: assertions={spec['assertions']}, variants={spec['variants']}")
    print(f"Modifiers: {spec['modifiers']}")
    print(f"\nPrompt:\n{render(spec)}")
    print(f"\nGold: {spec['positives'] or 'all NONE'}")
    
    # Generate with the model, capturing thinking
    response = client.messages.create(
        model=MODEL,
        max_tokens=16000,
        thinking={
            "type": "enabled",
            "budget_tokens": 10000
        },
        messages=[{
            "role": "user",
            "content": render(spec)
        }]
    )
    
    # Print thinking and response
    for block in response.content:
        if block.type == "thinking":
            print(f"\n[Model Thinking]:\n{block.thinking}")
        elif block.type == "text":
            print(f"\n[Model Output]:\n{block.text}")

In [ ]:


# ===========================================================================
# Cell 3: full run, 6000 specs. Roughly 8 minutes on the API.
# Only if the unlisted flag rate is at the floor and probe lift moved.
# ===========================================================================
GEN_FULL = "v18-full"
specs = make_specs(6000, vocab=vocab, seed=0)
generate_raw(specs, make_sample_fn(client, model=MODEL, reasoning_effort=EFFORT,
                                   api=API),
             gen_id=GEN_FULL, max_workers=16)

v18_id, stats = build_dataset_from_raw(
    GEN_FULL, resolver=v14_sample_to_instances, mode="markers",
    generator={"prompt_sha": fingerprint(), "model": MODEL, "version": "v18",
               "note": "assertion and modifier axes separated"},
    vocab_source=vocab.fingerprint(), seed=0, notes="v18, 6000 specs")
print(v18_id, stats["reject_reasons"])
v18, _ = load_dataset(v18_id)

v18_f, _ = frame_diag.attach_frames(v18, GEN_FULL)
print(f"nmi_sentence {frame_diag.label_purity(v18_f)['nmi_sentence']:.3f}")
print(f"probe lift   {gates.shortcut_probe(v18_f)[0]:.3f}")

comp = dv.compare(train_f, v14, match_size=True)[["measurement", "tier", "corpus",
                                                  "synth"]]
comp = comp.rename(columns={"synth": "v14"})
comp["v17"] = dv.compare(train_f, v17, match_size=True).synth.values
comp["v18"] = dv.compare(train_f, v18, match_size=True).synth.values
print(comp.to_string(index=False))



In [ ]:

# ===========================================================================
# Cell 4: train v18. Five seeds, not three.
#
# v17's seed sd is 0.035 with individual seeds at 0.365 and 0.430. At three seeds the
# standard error is near 0.020, so a difference has to clear roughly 0.06 to mean
# anything, and every generator comparison on record sits inside that. Five seeds is the
# minimum that can carry a 0.03 effect, and ten for anything that goes in the paper.
# ===========================================================================
import pandas as pd
from ddi.train import train_and_eval
from ddi.experiment import log_run

BASE = {"model_name": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
        "epochs": 3, "lr": 2e-5, "batch_size": 32, "max_length": 256,
        "neg_ratio": None, "render_mode": "markers"}


def train_arm(name, records, seeds=(0, 1, 2, 3, 4), notes=""):
    rows = []
    for seed in seeds:
        cfg = {**BASE, "seed": seed, "dataset": name}
        m = train_and_eval(cfg, records, dev)
        log_run(cfg, m, notes=notes or name)
        rows.append({"arm": name, "seed": seed, "f1": m["micro_f1_pos"],
                     "p": m["micro_p_pos"], "r": m["micro_r_pos"]})
        print(f"{name} seed={seed} f1={m['micro_f1_pos']:.3f} "
              f"p={m['micro_p_pos']:.3f} r={m['micro_r_pos']:.3f}")
    df = pd.DataFrame(rows)
    print(f"{name}: f1 {df.f1.mean():.3f} (sd {df.f1.std():.3f})  "
          f"P {df.p.mean():.3f}  R {df.r.mean():.3f}  n={len(df)}")
    return df


res = [train_arm("v18", v18)]
print("""
reference, same dev:
  human filtered      0.790   P 0.754  R 0.829
  v14 full            0.379   P 0.302  R 0.511
  v14 judged-pruned   0.408   P 0.304  R 0.623
  v17 (0959)          0.391   P 0.381  R 0.403   sd 0.035
  v15 full            0.333   P 0.272  R 0.430

v18 should move recall, not precision. v17 already reached P 0.381 and lost on R 0.403,
and the frame shortcut is a recall failure: real sentences carry no frame signature so
the classifier has nothing to fall back on. A v18 that gains precision and holds recall
means the shortcut was not what was costing recall.
""")


In [ ]:


# ===========================================================================
# Cell 5: verifier pruning on v18. Separate, because it is a separate claim.
#
# Pruning gained 0.041 on v14 and 0.042 on v15 against a random-prune control, with
# recall as the mechanism in both: v14 0.525 to 0.623, v15 0.453 to 0.534, precision
# flat. It has never been run on v17 or v18.
#
# v18 is the case where it might not replicate, and that is worth knowing either way. The
# verifier flags sentences whose surface asserts something the gold does not, which is
# precisely what the `unlisted` variant produces on purpose. If pruning helps v18 as much
# as it helped v14, some of that gain is the verifier deleting the hard negatives v18 was
# built to add. Check the flag rate on unlisted specs before reading the F1.
# ===========================================================================
from ddi.pruning import build_arms

VGEN_FULL = "v18-full-verify"
generate_raw(build_batches(v18), make_binary_verifier(client, model=MODEL),
             gen_id=VGEN_FULL, max_workers=8)
vdf_full = load_verdicts(VGEN_FULL)

variant_of = {f"synth:{GEN_FULL}:{s['spec_index']}": tuple(s["variants"]) or ("plain",)
              for s in make_specs(6000, vocab=vocab, seed=0)}
flag = Counter()
for r in vdf_full.itertuples():
    if r.gold_pos:
        continue
    v = variant_of.get(re.sub(r":s\d+$", "", r.sent_id), ("?",))[0]
    flag[(v, bool(r.flagged))] += 1
print(f"{'variant':<14} {'n_none':>8} {'flagged':>8} {'rate':>7}")
for v in sorted({k for k, _ in flag}):
    n = flag[(v, True)] + flag[(v, False)]
    print(f"{v:<14} {n:>8} {flag[(v, True)]:>8} {flag[(v, True)] / max(n, 1):>7.3f}")
print("if the unlisted rate is far above the others, pruning is removing v18's own "
      "hard negatives and the two interventions are working against each other")

arms = build_arms(v18, vdf_full, seed=0)
for name, recs in arms.items():
    print(f"{name:<20} {len(recs)} instances")
for name, recs in arms.items():
    res.append(train_arm(f"v18-{name}", recs, notes="v18 pruning arms"))


In [ ]:


# ===========================================================================
# Cell 6: union arms, size-controlled.
#
# v14 and v18 have complementary profiles: v14 is recall-heavy at P 0.302 R 0.511, v17
# and by construction v18 are precision-heavy. Different inventories, different failure
# modes, and no generation cost to combine them.
#
# The confound is size. A union is twice the data, and the mixing curve says synthetic
# saturates: worth +0.52 F1 at 100 human sentences and nothing past about 1,200. So a
# union that beats its parents may only be showing that more data helps, which is a
# claim about quantity, not about heterogeneity.
#
# The control is to subsample every arm to the same sentence count. `union-N` draws half
# its sentences from each parent and is the same size as `v14-N` and `v18-N`, so any gap
# is heterogeneity. `union-2N` is run as well, and the difference between union-N and
# union-2N is what quantity alone buys.
#
# v15 is included in the three-way arm only. It is the worst standalone at 0.333 but it
# was generated against corpus-matched content, so it is the most different from the
# other two, and heterogeneity is the whole hypothesis. If the three-way beats the
# two-way, being different matters more than being good, which is a result worth having.
# ===========================================================================
from ddi.mixing import subsample_by_sentence


def n_sentences(recs):
    return len({r["sent_id"] for r in recs})


def union(parts, n_sent_total, seed=0):
    """Equal sentence counts from each parent, summing to n_sent_total."""
    per = n_sent_total // len(parts)
    out = []
    for i, p in enumerate(parts):
        out += subsample_by_sentence(p, per, seed=seed + i)
    return out


N = min(n_sentences(v14), n_sentences(v18))
print(f"v14 {n_sentences(v14)} sentences, v18 {n_sentences(v18)}, matching at N={N}")

arms = {
    "v14-N":       subsample_by_sentence(v14, N, seed=0),
    "v18-N":       subsample_by_sentence(v18, N, seed=0),
    "union-N":     union([v14, v18], N, seed=0),
    "union-2N":    union([v14, v18], 2 * N, seed=0),
}
if V15_ID:
    v15, _ = load_dataset(V15_ID)
    arms["union3-N"] = union([v14, v18, v15], N, seed=0)
    arms["union3-3N"] = union([v14, v18, v15], 3 * N, seed=0)

for name, recs in arms.items():
    print(f"{name:<12} {n_sentences(recs):>6} sentences  {len(recs):>7} instances  "
          f"pos rate {sum(1 for r in recs if r['label'] != 'NONE') / len(recs):.3f}")

for name, recs in arms.items():
    res.append(train_arm(name, recs, notes="union arms, size-controlled"))

summary = pd.concat(res)
print(summary.groupby("arm")[["f1", "p", "r"]].agg(["mean", "std", "count"])
      .to_string())
print("""
  union-N beats both v14-N and v18-N        -> heterogeneity, and the paper's generator
     story becomes complementarity rather than a version sequence
  union-N matches its parents, union-2N wins -> quantity only, and the mixing curve
     already covers that ground
  union3-N beats union-N                     -> being different beats being good, and
     v15 is worth keeping in the corpus of generators rather than treated as a failure
""")